# Zinc-Ingot and Warehouse-Receipt Data Audit

This notebook reads raw data and never writes to `data/raw`. The physical scope accepts cash and cash-matching contracts. Positive activity requires `Quantity > 0` and `Price > 0` in the physical market and `TradesVolume > 0` in the certificate market. Zinc dust is excluded from the ingot-grade analysis.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

def find_project_dir() -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        direct = base / "data/raw/physical/zinc_physical_raw.csv"
        nested = base / "commodity/zinc/data/raw/physical/zinc_physical_raw.csv"
        if direct.exists():
            return base
        if nested.exists():
            return base / "commodity/zinc"
    raise FileNotFoundError("Could not locate commodity/zinc from the current directory")

PROJECT_DIR = find_project_dir()
PHYSICAL_PATH = PROJECT_DIR / "data/raw/physical/zinc_physical_raw.csv"
CERTIFICATE_PATH = PROJECT_DIR / "data/raw/certificate/zinc_certificate_raw.csv"

physical_raw = pd.read_csv(PHYSICAL_PATH, encoding="utf-8-sig", low_memory=False)
certificate_raw = pd.read_csv(CERTIFICATE_PATH, encoding="utf-8-sig", low_memory=False)

print(f"Project: {PROJECT_DIR}")
print(f"Physical raw: {physical_raw.shape[0]:,} rows × {physical_raw.shape[1]} columns")
print(f"Certificate raw: {certificate_raw.shape[0]:,} rows × {certificate_raw.shape[1]} columns")

## Preparation and definition of positive transaction
A common Jalali date is constructed because the physical market uses `date` while the certificate uses `PersianDate`. Ingot grade is extracted from `GoodsName` and normalized to two decimals, such as `99.97`.

In [ ]:
def normalize_fa(value: object) -> str:
    return str(value or "").replace("ي", "ی").replace("ك", "ک").strip()

def normalize_jalali_date(value: object) -> str | None:
    if pd.isna(value):
        return None
    parts = str(value).strip().replace("-", "/").split("/")
    if len(parts) != 3:
        return None
    try:
        year, month, day = map(int, parts)
    except ValueError:
        return None
    return f"{year:04d}/{month:02d}/{day:02d}"

def extract_zinc_grade(goods_name: object) -> str | None:
    name = normalize_fa(goods_name)
    if "شمش روی" not in name:
        return None
    match = re.search(r"(\d{2}(?:\.\d+)?)", name)
    return f"{float(match.group(1)):.2f}" if match else None

physical = physical_raw.copy()
certificate = certificate_raw.copy()

for column in ["Quantity", "Price", "TotalPrice"]:
    physical[column] = pd.to_numeric(physical[column], errors="coerce")
for column in ["TradesVolume", "TradesValue", "TodaySettlementPrice"]:
    certificate[column] = pd.to_numeric(certificate[column], errors="coerce")

physical["jalali_date"] = physical["date"].map(normalize_jalali_date)
certificate["jalali_date"] = certificate["PersianDate"].map(normalize_jalali_date)
physical["grade"] = physical["GoodsName"].map(extract_zinc_grade)

ALLOWED_CONTRACT_TYPES = {"نقدی", "نقدی (مچینگ)"}
physical["contract_type_normalized"] = physical["ContractType"].map(normalize_fa)
physical_trades = physical.loc[
    physical["contract_type_normalized"].isin(ALLOWED_CONTRACT_TYPES)
    & (physical["Quantity"] > 0)
    & (physical["Price"] > 0)
].copy()
ingot_trades = physical_trades.loc[physical_trades["grade"].notna()].copy()
certificate_trades = certificate.loc[certificate["TradesVolume"] > 0].copy()

assert set(physical_trades["contract_type_normalized"].unique()) <= ALLOWED_CONTRACT_TYPES
assert ingot_trades["GoodsName"].map(normalize_fa).str.contains("خاک روی", na=False).sum() == 0
assert certificate_trades["jalali_date"].notna().all()

print(f"Allowed physical contracts: {sorted(ALLOWED_CONTRACT_TYPES)}")
print(f"Positive physical rows (cash and cash-matching): {len(physical_trades):,}")
print(f"Positive zinc-ingot rows: {len(ingot_trades):,}")
print(f"Positive certificate days: {certificate_trades['jalali_date'].nunique():,}")

## Overview of two markets and trading days
The first table shows the size and coverage of each market. The second table is the complete calendar of active days; So it can be seen that on any given date only the physical market, only the certificate, or both were active.

In [ ]:
market_overview = pd.DataFrame([
    {
        "market": "physical — all zinc-related goods (cash + cash-matching)",
        "positive_trade_rows": len(physical_trades),
        "trading_days": physical_trades["jalali_date"].nunique(),
        "first_trade_day": physical_trades["jalali_date"].min(),
        "last_trade_day": physical_trades["jalali_date"].max(),
        "total_volume": physical_trades["Quantity"].sum(),
    },
    {
        "market": "physical — zinc ingot only (cash + cash-matching)",
        "positive_trade_rows": len(ingot_trades),
        "trading_days": ingot_trades["jalali_date"].nunique(),
        "first_trade_day": ingot_trades["jalali_date"].min(),
        "last_trade_day": ingot_trades["jalali_date"].max(),
        "total_volume": ingot_trades["Quantity"].sum(),
    },
    {
        "market": "zinc certificate",
        "positive_trade_rows": len(certificate_trades),
        "trading_days": certificate_trades["jalali_date"].nunique(),
        "first_trade_day": certificate_trades["jalali_date"].min(),
        "last_trade_day": certificate_trades["jalali_date"].max(),
        "total_volume": certificate_trades["TradesVolume"].sum(),
    },
]).set_index("market")
display(market_overview)

physical_all_days = set(physical_trades["jalali_date"].dropna())
physical_ingot_days = set(ingot_trades["jalali_date"].dropna())
certificate_days = set(certificate_trades["jalali_date"].dropna())
all_active_days = sorted(physical_all_days | certificate_days)

market_calendar = pd.DataFrame({"jalali_date": all_active_days})
market_calendar["physical_any_zinc_trade"] = market_calendar["jalali_date"].isin(physical_all_days)
market_calendar["physical_ingot_trade"] = market_calendar["jalali_date"].isin(physical_ingot_days)
market_calendar["certificate_trade"] = market_calendar["jalali_date"].isin(certificate_days)
market_calendar["both_ingot_and_certificate"] = (
    market_calendar["physical_ingot_trade"] & market_calendar["certificate_trade"]
)
display(market_calendar)

In [ ]:
# Monthly view of activity density and gaps in both markets
monthly_activity = (
    market_calendar.assign(jalali_month=lambda frame: frame["jalali_date"].str[:7])
    .groupby("jalali_month", as_index=False)
    .agg(
        physical_any_zinc_days=("physical_any_zinc_trade", "sum"),
        physical_ingot_days=("physical_ingot_trade", "sum"),
        certificate_days=("certificate_trade", "sum"),
        both_ingot_certificate_days=("both_ingot_and_certificate", "sum"),
    )
)
display(monthly_activity)

## Positive trades and trading dates by ingot grade

`trade_rows` counts positive source rows and `trading_days` counts unique active dates. `trade_dates` preserves the exact activity calendar for each grade.

In [ ]:
grade_activity = (
    ingot_trades.groupby("grade", as_index=False)
    .agg(
        trade_rows=("grade", "size"),
        trading_days=("jalali_date", "nunique"),
        first_trade_day=("jalali_date", "min"),
        last_trade_day=("jalali_date", "max"),
        total_quantity=("Quantity", "sum"),
        trade_dates=("jalali_date", lambda values: sorted(set(values.dropna()))),
    )
    .sort_values(["trading_days", "trade_rows"], ascending=False)
    .reset_index(drop=True)
)
display(grade_activity)

# Optional date-level view with one observation per row
grade_trade_dates = (
    ingot_trades[["grade", "jalali_date"]]
    .drop_duplicates()
    .sort_values(["grade", "jalali_date"])
    .reset_index(drop=True)
)
display(grade_trade_dates)

## Certificate trading days and overlap with the target rate
To add `99.96` later, just change the value of `TARGET_GRADES` to `['99.97', '99.96']`. The overlap table shows the exact date of activity of each market and the summary of the number of common days.

In [ ]:
certificate_trade_dates = (
    certificate_trades[["jalali_date", "DT", "ContractCode", "TradesVolume", "TodaySettlementPrice"]]
    .sort_values("jalali_date")
    .reset_index(drop=True)
)
display(certificate_trade_dates)

TARGET_GRADES = ["99.97"]  # later: ["99.97", "99.96"]

target_days_by_grade = {
    grade: set(ingot_trades.loc[ingot_trades["grade"].eq(grade), "jalali_date"].dropna())
    for grade in TARGET_GRADES
}
overlap_dates = sorted(certificate_days | set().union(*target_days_by_grade.values()))
overlap_calendar = pd.DataFrame({"jalali_date": overlap_dates})
overlap_calendar["certificate_trade"] = overlap_calendar["jalali_date"].isin(certificate_days)
for grade, grade_days in target_days_by_grade.items():
    overlap_calendar[f"ingot_{grade}_trade"] = overlap_calendar["jalali_date"].isin(grade_days)
    overlap_calendar[f"certificate_and_{grade}"] = (
        overlap_calendar["certificate_trade"] & overlap_calendar[f"ingot_{grade}_trade"]
    )

overlap_summary = pd.DataFrame([
    {
        "grade": grade,
        "ingot_trading_days": len(grade_days),
        "certificate_trading_days": len(certificate_days),
        "common_trading_days": len(grade_days & certificate_days),
        "common_dates": sorted(grade_days & certificate_days),
    }
    for grade, grade_days in target_days_by_grade.items()
])
display(overlap_summary)
display(overlap_calendar)

## The suppliers of every caliber and their share of transactions
The supplier name is taken from `ArzehKonandeh` and if empty, `ProducerName` is used. `trade_count_share_pct` is the company's share of the number of positive trade rows of the same grade and `quantity_share_pct` is its share of the traded volume of the same grade. These two criteria are intentionally separate.

In [ ]:
supplier = ingot_trades["ArzehKonandeh"].fillna("").astype(str).str.strip()
producer = ingot_trades["ProducerName"].fillna("").astype(str).str.strip()
ingot_trades["supplier"] = supplier.where(supplier.ne(""), producer)

supplier_by_grade = (
    ingot_trades.groupby(["grade", "supplier"], as_index=False)
    .agg(
        trade_rows=("supplier", "size"),
        trading_days=("jalali_date", "nunique"),
        total_quantity=("Quantity", "sum"),
        first_trade_day=("jalali_date", "min"),
        last_trade_day=("jalali_date", "max"),
    )
)
supplier_by_grade["trade_count_share_pct"] = (
    100 * supplier_by_grade["trade_rows"]
    / supplier_by_grade.groupby("grade")["trade_rows"].transform("sum")
)
supplier_by_grade["quantity_share_pct"] = (
    100 * supplier_by_grade["total_quantity"]
    / supplier_by_grade.groupby("grade")["total_quantity"].transform("sum")
)
supplier_by_grade[["trade_count_share_pct", "quantity_share_pct"]] = (
    supplier_by_grade[["trade_count_share_pct", "quantity_share_pct"]].round(2)
)
supplier_by_grade = supplier_by_grade.sort_values(
    ["grade", "quantity_share_pct", "trade_count_share_pct"],
    ascending=[True, False, False],
).reset_index(drop=True)
display(supplier_by_grade)

# Validation: supplier shares within each positive-volume grade should sum to approximately 100%.
supplier_share_check = (
    supplier_by_grade.groupby("grade", as_index=False)
    .agg(
        suppliers=("supplier", "nunique"),
        trade_count_share_pct=("trade_count_share_pct", "sum"),
        quantity_share_pct=("quantity_share_pct", "sum"),
    )
)
display(supplier_share_check)

## Focus on 99.97 and 99.98 bars
This section calculates daily prices for each grade and for the `99.97 + 99.98` basket using transaction-volume weights (`Quantity`). It separately reports the incremental date coverage from adding grade 99.98 and the resulting increase in overlap with the certificate. The scope remains limited to cash and cash-matching contracts.

In [ ]:
FOCUS_GRADES = ["99.97", "99.98"]
focus_trades = ingot_trades.loc[ingot_trades["grade"].isin(FOCUS_GRADES)].copy()

# Audit source units before applying any conversion.
unit_audit = (
    focus_trades.groupby(["grade", "Currency", "Unit"], dropna=False, as_index=False)
    .agg(trade_rows=("grade", "size"), total_quantity=("Quantity", "sum"))
)
display(unit_audit)

# در داده IME ستون Price به‌عنوان قیمت گزارش‌شده هر کیلو استفاده می‌شود؛
# اگر سند قرارداد بعداً ضریب دیگری نشان داد، فقط این ضریب را تغییر بده.
PHYSICAL_PRICE_TO_PER_KG = 1.0

focus_trades["weighted_value"] = focus_trades["Price"] * focus_trades["Quantity"]
daily_grade_price = (
    focus_trades.groupby(["jalali_date", "grade"], as_index=False)
    .agg(
        trade_rows=("grade", "size"),
        quantity=("Quantity", "sum"),
        weighted_value=("weighted_value", "sum"),
        suppliers=("supplier", "nunique"),
    )
)
daily_grade_price["price_per_kg"] = (
    daily_grade_price["weighted_value"] / daily_grade_price["quantity"]
    * PHYSICAL_PRICE_TO_PER_KG
)
daily_grade_price = daily_grade_price.drop(columns="weighted_value").sort_values(
    ["jalali_date", "grade"]
)
display(daily_grade_price)

In [ ]:
# Daily volume-weighted price of the two-grade basket
daily_basket_price = (
    focus_trades.groupby("jalali_date", as_index=False)
    .agg(
        trade_rows=("grade", "size"),
        grades_traded=("grade", lambda values: sorted(set(values))),
        quantity=("Quantity", "sum"),
        weighted_value=("weighted_value", "sum"),
        suppliers=("supplier", "nunique"),
    )
)
daily_basket_price["basket_price_per_kg"] = (
    daily_basket_price["weighted_value"] / daily_basket_price["quantity"]
    * PHYSICAL_PRICE_TO_PER_KG
)
daily_basket_price = daily_basket_price.drop(columns="weighted_value").sort_values("jalali_date")
display(daily_basket_price)

grade_days = {
    grade: set(daily_grade_price.loc[daily_grade_price["grade"].eq(grade), "jalali_date"])
    for grade in FOCUS_GRADES
}
days_97 = grade_days["99.97"]
days_98 = grade_days["99.98"]
basket_days = days_97 | days_98

coverage_97_98 = pd.DataFrame([
    {"metric": "99.97 trading days", "days": len(days_97)},
    {"metric": "99.98 trading days", "days": len(days_98)},
    {"metric": "days with both grades", "days": len(days_97 & days_98)},
    {"metric": "basket union days", "days": len(basket_days)},
    {"metric": "new physical days added by 99.98 to 99.97", "days": len(days_98 - days_97)},
    {"metric": "certificate ∩ 99.97 days", "days": len(certificate_days & days_97)},
    {"metric": "certificate ∩ basket days", "days": len(certificate_days & basket_days)},
    {
        "metric": "new certificate-overlap days added by 99.98",
        "days": len((certificate_days & basket_days) - (certificate_days & days_97)),
    },
])
display(coverage_97_98)

added_physical_dates = sorted(days_98 - days_97)
added_certificate_overlap_dates = sorted((certificate_days & basket_days) - (certificate_days & days_97))
display(pd.DataFrame({"days_added_to_physical_series": pd.Series(added_physical_dates)}))
display(pd.DataFrame({"days_added_to_certificate_overlap": pd.Series(added_certificate_overlap_dates)}))

## Price difference between grades 99.97 and 99.98

Direct comparison is restricted to dates when both grades trade. A positive difference means the volume-weighted 99.98 price exceeds the 99.97 price.

In [ ]:
price_wide = daily_grade_price.pivot(index="jalali_date", columns="grade", values="price_per_kg")
quantity_wide = daily_grade_price.pivot(index="jalali_date", columns="grade", values="quantity")
both_grade_price_comparison = price_wide.dropna(subset=FOCUS_GRADES).copy()
both_grade_price_comparison["difference_98_minus_97"] = (
    both_grade_price_comparison["99.98"] - both_grade_price_comparison["99.97"]
)
both_grade_price_comparison["difference_pct_vs_97"] = (
    100 * both_grade_price_comparison["difference_98_minus_97"]
    / both_grade_price_comparison["99.97"]
)
both_grade_price_comparison = both_grade_price_comparison.reset_index()
display(both_grade_price_comparison)

price_difference_summary = both_grade_price_comparison[[
    "difference_98_minus_97", "difference_pct_vs_97"
]].describe().T
display(price_difference_summary)

## Certificate: today's settlement price and control value divided by volume
The price used for the certificate is `TodaySettlementPrice` only. The calculated column below simply controls what `TradesValue / TradesVolume` is equal to and does not replace the original price.

In [ ]:
certificate_daily_price = certificate_trades[[
    "jalali_date", "DT", "TradesVolume", "TradesValue", "TodaySettlementPrice"
]].copy()
certificate_daily_price["value_divided_by_volume"] = (
    certificate_daily_price["TradesValue"] / certificate_daily_price["TradesVolume"]
)
certificate_daily_price["settlement_check_difference"] = (
    certificate_daily_price["value_divided_by_volume"]
    - certificate_daily_price["TodaySettlementPrice"]
)
certificate_daily_price["settlement_check_ok"] = np.isclose(
    certificate_daily_price["value_divided_by_volume"],
    certificate_daily_price["TodaySettlementPrice"],
    rtol=1e-7,
    atol=0.51,
)
display(certificate_daily_price.sort_values("jalali_date"))
display(certificate_daily_price[["settlement_check_difference"]].describe().T)

basket_vs_certificate = (
    daily_basket_price.merge(
        certificate_daily_price[["jalali_date", "TodaySettlementPrice"]],
        on="jalali_date",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("jalali_date")
)
basket_vs_certificate["certificate_minus_basket"] = (
    basket_vs_certificate["TodaySettlementPrice"]
    - basket_vs_certificate["basket_price_per_kg"]
)
basket_vs_certificate["certificate_premium_pct"] = (
    100 * basket_vs_certificate["certificate_minus_basket"]
    / basket_vs_certificate["basket_price_per_kg"]
)
display(basket_vs_certificate)

## Graph of price, difference, volume and number of active days

In [ ]:
plot_prices = price_wide.reindex(columns=FOCUS_GRADES).copy()
plot_prices['99.97 + 99.98 basket'] = daily_basket_price.set_index('jalali_date')['basket_price_per_kg']
plot_prices = plot_prices.sort_index()
difference_plot = both_grade_price_comparison.set_index('jalali_date')['difference_pct_vs_97']
fig = make_subplots(rows=3, cols=1, vertical_spacing=0.08,
    subplot_titles=('Daily volume-weighted zinc-ingot prices',
                    '99.98 premium / discount versus 99.97', 'Daily traded quantity by grade'))
for column, color in zip(plot_prices.columns, ['#1976D2', '#00897B', '#EF6C00']):
    fig.add_trace(go.Scatter(x=plot_prices.index, y=plot_prices[column],
        name=column, line_color=color), row=1, col=1)
fig.add_trace(go.Bar(x=difference_plot.index, y=difference_plot,
    marker_color=np.where(difference_plot.ge(0), '#2E7D32', '#C62828'),
    name='Grade difference'), row=2, col=1)
for column, color in zip(FOCUS_GRADES, ['#1976D2', '#00897B']):
    fig.add_trace(go.Bar(x=quantity_wide.index, y=quantity_wide[column],
        name=f'{column} quantity', marker_color=color), row=3, col=1)
fig.add_hline(y=0, line_color='#455A64', row=2, col=1)
fig.update_yaxes(title_text='Reported price per kg', row=1, col=1)
fig.update_yaxes(title_text='Difference (%)', row=2, col=1)
fig.update_yaxes(title_text='Source quantity', row=3, col=1)
fig.update_layout(height=1000, barmode='stack', template='plotly_white')
fig.show()

activity_dates = sorted(basket_days | certificate_days)
activity_matrix = pd.DataFrame([
    [date in days_97 for date in activity_dates],
    [date in days_98 for date in activity_dates],
    [date in basket_days for date in activity_dates],
    [date in certificate_days for date in activity_dates],
], index=['99.97', '99.98', 'basket', 'certificate'], columns=activity_dates, dtype=int)
display(activity_matrix)
fig = go.Figure(go.Heatmap(z=activity_matrix.to_numpy(), x=activity_dates,
    y=activity_matrix.index, zmin=0, zmax=1, colorscale='Blues',
    hovertemplate='%{y}<br>%{x}<br>Traded: %{z}<extra></extra>'))
fig.update_layout(title='Trading-day availability', xaxis_title='Jalali date',
    height=350, template='plotly_white')
fig.show()

## Standard market dashboard

This governed, read-only section uses the same presentation contract across commodity projects:
source coverage, physical and certificate activity, separate price panels, physical-goods
composition, and validated processed bubbles. It never writes raw data or constructs a missing
bubble. For Zinc, product comparability still follows the project-specific workflow.

In [ ]:
from pathlib import Path
import sys

def locate_workspace(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "commodity" / "zinc").exists() and (candidate / "shared").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root")

WORKSPACE_ROOT = locate_workspace()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from shared.notebook_tools.commodity_dashboard import (
    goods_type_counts,
    load_markets,
    market_summary,
    plot_available_bubbles,
    plot_goods_type_counts,
    plot_market_prices,
    plot_trade_activity,
)

PROJECT_DIR = WORKSPACE_ROOT / "commodity" / "zinc"
physical_dashboard, certificate_dashboard = load_markets(
    PROJECT_DIR, "zinc", physical_filename=None
)
display(market_summary(physical_dashboard, certificate_dashboard))
plot_trade_activity(physical_dashboard, certificate_dashboard, "Zinc")
plot_market_prices(physical_dashboard, certificate_dashboard, "Zinc")
goods_count_table = plot_goods_type_counts(physical_dashboard, "Zinc", top_n=30)
display(goods_count_table)
bubble_series_plotted = plot_available_bubbles(PROJECT_DIR, "Zinc")

## Signed distribution and historical percentile ranks
The histogram retains positive and negative bubbles. The timeline compares equal-weight and exponentially recent-weighted expanding ranks.
For each date t, rank = 100 ? sum(w ? 1[bubble <= bubble_t]) / sum(w), using dates through t, including the current point and all ties. Equal weights are 1; recent weights are 2^(-age_in_calendar_days / 90). The half-life is configurable in the distribution builder.
The first rank is 100 by definition; small histories are unstable. Hover shows history size and effective weighted count. These ranks are descriptive, not reversal probabilities or trading thresholds.
Raw signed bubble values are not smoothed. Observed and interpolated points remain included and labeled in the table. Ranking uses no future rows, but interpolated source bubbles may depend on later anchors: this is not a vintage-safe backtest. The histogram uses the full currently loaded sample.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'shared').is_dir() and (p / 'commodity/zinc').is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from shared.market_analysis.bubble_percentiles import figures
ranks = pd.read_csv(root / 'commodity/zinc/data/processed/bubble/zinc_bubble_distribution.csv', parse_dates=['observation_date'])
for series_id in ['certificate_physical', 'certificate_intrinsic', 'physical_intrinsic']:
    series = ranks.loc[ranks.series_id.eq(series_id)].sort_values('observation_date')
    for figure in figures(series, 'Zinc: ' + series.comparison.iloc[0]):
        figure.show()
    display(series[['observation_date','bubble_pct','expanding_percentile','recent_weighted_percentile','history_count','effective_weighted_count','point_method']].tail(10))
